In [6]:
import pandas as pd # Import "pandas" could not be resolved from source
from pathlib import Path

#Load the customer reviews dataset

file_path = Path("UrbanKart_HuggingFace_NLP_Customer_Reviews.csv")

df = pd.read_csv(file_path)

In [18]:
df.head()

,Customer_ID,review_text,sentiment_label,satisfaction_score,preferred_category,preferred_channel,customer_value_segment,customer_tenure_months,orders_last_12m,days_since_last_purchase,support_tickets_12m,return_rate,churn,purchase_next_30_days,data_origin,split
0,CUST009609,I am unhappy with my Sports & Fitness order an...,negative,2.00,Sports & Fitness,Website,Low Value,26,8,84,4,0.190,1,0,synthetic_generated_for_training,train
1,CUST004601,I was disappointed with my Electronics purchas...,negative,1.56,Electronics,Mobile App,Medium Value,10,7,63,4,0.000,1,1,synthetic_generated_for_training,test
2,CUST000722,I was disappointed with my Sports & Fitness pu...,negative,1.89,Sports & Fitness,Website,Low Value,12,6,73,6,0.119,1,0,synthetic_generated_for_training,train
3,CUST007179,The Books shopping experience needs improvemen...,negative,2.41,Books,Physical Store,Low Value,12,4,322,0,0.210,1,0,synthetic_generated_for_training,train
4,CUST002160,"My Sports & Fitness order was frustrating, and...",negative,2.46,Sports & Fitness,Marketplace,Medium Value,20,6,36,2,0.327,0,1,synthetic_generated_for_training,validation


In [7]:
print("Dataset shape:", df.shape)

print("\nColumn names:")

print(df.columns.tolist())

Dataset shape: (6000, 16)

Column names:
['Customer_ID', 'review_text', 'sentiment_label', 'satisfaction_score', 'preferred_category', 'preferred_channel', 'customer_value_segment', 'customer_tenure_months', 'orders_last_12m', 'days_since_last_purchase', 'support_tickets_12m', 'return_rate', 'churn', 'purchase_next_30_days', 'data_origin', 'split']


In [8]:
print("\nFirst five reveiws:")
display(df[["review_text","sentiment_label","split"]].head())


First five reveiws:


,review_text,sentiment_label,split
0,I am unhappy with my Sports & Fitness order an...,negative,train
1,I was disappointed with my Electronics purchas...,negative,test
2,I was disappointed with my Sports & Fitness pu...,negative,train
3,The Books shopping experience needs improvemen...,negative,train
4,"My Sports & Fitness order was frustrating, and...",negative,validation


In [9]:
print("\nSentiment distribution:")
print(df["sentiment_label"].value_counts())


Sentiment distribution:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [10]:
print("\nMissing review text:",df["review_text"].isna().sum())
print("Missing sentiment labels:",df["sentiment_label"].isna().sum())


Missing review text: 0
Missing sentiment labels: 0


In [11]:
pip install transformers 

Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip show transformers

Name: transformers
Version: 5.19.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: c:\Users\user\.conda\envs\regression_app\Lib\site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [13]:
from transformers import pipeline

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME
)

print("Sentiment model loaded successfully!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13394.16it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded successfully!


In [14]:
test_reviews = [
"The product quality is excellent and delivery was fast.",
"The product arrived damaged and customer support was unhelpful.",
"The package arrived yesterday."]

for review in test_reviews:
     result=sentiment_analyzer(review)[0]

print("Review:", review)
print("Sentiment:", result["label"])
print("Score:", round (result["score"], 4))
print("-"*60)

Review: The package arrived yesterday.
Sentiment: neutral
Score: 0.5292
------------------------------------------------------------


In [ ]:
print("Dataset split distribution:")
print(df["split"].value_counts())

print("\nsentiment labels:")
print(df["sentiment_label"].value_counts())

Dataset split distribution:
split
train         4800
test           600
validation     600
Name: count, dtype: int64

sentiment labels:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [17]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Select held-out test data only
test_df = df.loc[
    df["split"].str.lower() == "test",
    ["review_text", "sentiment_label"]
].dropna().copy()

print("Test records:", len(test_df))

# Predict in batches
predictions = sentiment_analyzer(
    test_df["review_text"].astype(str).tolist(),
    batch_size=16,
    truncation=True
)
test_df["predicted_sentiment"]=[p["label"].lower() for p in predictions]

Test records: 600
